In [1]:
# CELL 1: Install dependencies cleanly
!pip install -q "numpy<2.0.0" "pandas==2.2.2"
!pip install -q faiss-cpu sentence-transformers langchain langchain-community openai pypdf fpdf2 pyyaml gradio
!pip install -q tree-sitter==0.21.3 tree-sitter-languages==1.10.2

print("✅ Installation complete! Please restart your session before proceeding.")

✅ Installation complete! Please restart your session before proceeding.


In [2]:
# CELL 2: Core imports and global configuration
# Set your OpenAI API key here (or leave blank and set via environment variable / Colab secret).
# This wrapper is designed so you can swap OpenAI for Llama/Mistral later without touching downstream code.

import os
import re
import io
import json
import zipfile
import shutil
import ast as py_ast
import uuid
import time
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from fpdf import FPDF
import yaml

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------

# Set your OpenAI API key. In Colab: Runtime > you can also use `from google.colab import userdata`
# and `userdata.get('OPENAI_API_KEY')` if stored as a Colab secret.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")  # <-- put your key in Colab secrets or here
LLM_MODEL_NAME = "gpt-4o-mini"      # Swap this for a local Llama/Mistral endpoint later
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"

# Working directories (Colab local filesystem)
WORKDIR = "/content/rag_test_generator"
UPLOAD_DIR = os.path.join(WORKDIR, "uploads")
INDEX_DIR = os.path.join(WORKDIR, "index")
EXPORT_DIR = os.path.join(WORKDIR, "exports")

for d in [WORKDIR, UPLOAD_DIR, INDEX_DIR, EXPORT_DIR]:
    os.makedirs(d, exist_ok=True)

# Chunking configuration
CHUNK_SIZE_CHARS = 1200
CHUNK_OVERLAP_CHARS = 150

# Retrieval configuration
TOP_K_RETRIEVAL = 6

print(f"Workdir ready at: {WORKDIR}")
print(f"LLM model: {LLM_MODEL_NAME} | Embedding model: {EMBEDDING_MODEL_NAME}")
print("Set OPENAI_API_KEY before running generation cells (Colab secrets recommended).")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Workdir ready at: /content/rag_test_generator
LLM model: gpt-4o-mini | Embedding model: BAAI/bge-large-en-v1.5
Set OPENAI_API_KEY before running generation cells (Colab secrets recommended).


In [3]:
# CELL 3: Shared data structures used across the pipeline
# A unified "Chunk" schema lets requirements, code, and API docs live in one vector index
# while retaining metadata for traceability.

@dataclass
class ArtifactChunk:
    chunk_id: str
    source_type: str        # "requirement" | "code" | "api_doc"
    source_name: str        # filename or symbol name
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Requirement:
    req_id: str
    text: str
    source_file: str

@dataclass
class GeneratedTest:
    test_id: str
    requirement_id: str
    category: str           # unit | integration | edge | boundary | security
    title: str
    steps: str
    expected_result: str
    retrieved_context_ids: List[str] = field(default_factory=list)

# In-memory session state (reset by re-running this cell)
SESSION = {
    "requirements": [],      # List[Requirement]
    "chunks": [],             # List[ArtifactChunk]
    "faiss_index": None,      # faiss.Index
    "chunk_id_map": [],       # list mapping row -> chunk_id (parallel to faiss vectors)
    "generated_tests": [],    # List[GeneratedTest]
}

print("Data structures and session state initialized.")

Data structures and session state initialized.


In [4]:
# CELL 4: Upload and parse the Software Requirement Specification (SRS)
# Supports PDF and TXT. Splits into individual requirement statements using
# common numbering patterns (e.g., "FR1.", "1.", "REQ-01:") with a sentence-based fallback.

from google.colab import files as colab_files

def extract_text_from_pdf(filepath: str) -> str:
    reader = PdfReader(filepath)
    pages_text = []
    for page in reader.pages:
        page_text = page.extract_text() or ""
        pages_text.append(page_text)
    return "\n".join(pages_text)

def extract_text_from_txt(filepath: str) -> str:
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def split_requirements(raw_text: str, source_file: str) -> List[Requirement]:
    """
    Splits SRS text into discrete requirement statements.
    Recognizes patterns like 'FR1.', 'NFR2.', 'REQ-01:', '1.2.', or falls back to
    line-based splitting for unstructured documents.
    """
    pattern = re.compile(
        r"(?:^|\n)\s*((?:FR|NFR|REQ)[-\s]?\d+[a-zA-Z]?\.?|\d+\.\d*\.?)\s*(.+?)(?=\n\s*(?:FR|NFR|REQ)[-\s]?\d+|\n\s*\d+\.\d*\.|\Z)",
        re.DOTALL | re.IGNORECASE
    )
    matches = pattern.findall(raw_text)

    requirements = []
    if matches:
        for idx, (label, body) in enumerate(matches):
            clean_body = " ".join(body.strip().split())
            if len(clean_body) < 5:
                continue
            requirements.append(Requirement(
                req_id=f"REQ-{idx+1:03d}",
                text=f"{label.strip()} {clean_body}".strip(),
                source_file=source_file
            ))
    else:
        # Fallback: treat each non-empty line as a candidate requirement
        lines = [l.strip() for l in raw_text.split("\n") if len(l.strip()) > 15]
        for idx, line in enumerate(lines):
            requirements.append(Requirement(
                req_id=f"REQ-{idx+1:03d}",
                text=line,
                source_file=source_file
            ))
    return requirements

def upload_srs_file() -> List[Requirement]:
    """
    Prompts an interactive Colab upload dialog for the SRS file (PDF or TXT),
    parses it, and stores requirements into SESSION['requirements'].
    """
    print("Please select your SRS file (PDF or TXT)...")
    uploaded = colab_files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")

    filename = list(uploaded.keys())[0]
    filepath = os.path.join(UPLOAD_DIR, filename)
    with open(filepath, "wb") as f:
        f.write(uploaded[filename])

    if filename.lower().endswith(".pdf"):
        raw_text = extract_text_from_pdf(filepath)
    elif filename.lower().endswith(".txt"):
        raw_text = extract_text_from_txt(filepath)
    else:
        raise ValueError("Unsupported SRS format. Please upload a .pdf or .txt file.")

    reqs = split_requirements(raw_text, filename)
    SESSION["requirements"] = reqs
    print(f"Parsed {len(reqs)} requirements from '{filename}'.")
    for r in reqs[:5]:
        print(f"  {r.req_id}: {r.text[:90]}...")
    return reqs

# NOTE: This cell defines functions only. Call upload_srs_file() in a later
# interactive cell (or via the Gradio UI in Cell 14) to trigger the upload dialog.
print("SRS ingestion utilities ready. Call upload_srs_file() to upload your SRS.")

SRS ingestion utilities ready. Call upload_srs_file() to upload your SRS.


In [5]:
# CELL 5: Upload and parse source code (.zip archive)
# Extracts the archive, then parses Python files via the `ast` module and
# JS/TS/Java files via a lightweight regex-based function/class extractor
# (keeps the pipeline dependency-light while still giving RAG useful chunks).

def parse_python_file(filepath: str) -> List[ArtifactChunk]:
    chunks = []
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        source = f.read()
    try:
        tree = py_ast.parse(source)
    except SyntaxError:
        return chunks

    rel_name = os.path.basename(filepath)
    for node in py_ast.walk(tree):
        if isinstance(node, (py_ast.FunctionDef, py_ast.AsyncFunctionDef, py_ast.ClassDef)):
            try:
                snippet = py_ast.get_source_segment(source, node) or ""
            except Exception:
                snippet = ""
            if not snippet:
                continue
            docstring = py_ast.get_docstring(node) or ""
            chunks.append(ArtifactChunk(
                chunk_id=str(uuid.uuid4()),
                source_type="code",
                source_name=f"{rel_name}::{node.name}",
                text=f"{node.name}\n{docstring}\n{snippet}"[:CHUNK_SIZE_CHARS],
                metadata={"file": rel_name, "symbol": node.name, "kind": type(node).__name__}
            ))
    return chunks

def parse_generic_code_file(filepath: str) -> List[ArtifactChunk]:
    """
    Regex-based function/class extraction for JS, TS, Java, and other C-like languages.
    Used as a lightweight fallback so the pipeline doesn't hard-depend on full
    Tree-sitter grammar builds inside Colab.
    """
    chunks = []
    rel_name = os.path.basename(filepath)
    try:
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            source = f.read()
    except Exception:
        return chunks

    pattern = re.compile(
        r"((?:export\s+)?(?:async\s+)?function\s+\w+\s*\([^)]*\)\s*\{|"
        r"class\s+\w+\s*\{|"
        r"(?:public|private|protected)\s+[\w<>\[\]]+\s+\w+\s*\([^)]*\)\s*\{)"
    )
    matches = list(pattern.finditer(source))
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else min(start + CHUNK_SIZE_CHARS, len(source))
        snippet = source[start:end][:CHUNK_SIZE_CHARS]
        name_guess = re.sub(r"\s+", " ", m.group(0))[:60]
        chunks.append(ArtifactChunk(
            chunk_id=str(uuid.uuid4()),
            source_type="code",
            source_name=f"{rel_name}::{name_guess}",
            text=snippet,
            metadata={"file": rel_name, "symbol": name_guess}
        ))
    return chunks

def upload_and_parse_source_code(zip_path: Optional[str] = None) -> List[ArtifactChunk]:
    """
    Uploads a .zip of source code (or uses zip_path if already on disk),
    extracts it, and parses each file into ArtifactChunks stored in SESSION['chunks'].
    """
    if zip_path is None:
        print("Please select your source code .zip file...")
        uploaded = colab_files.upload()
        if not uploaded:
            raise ValueError("No file uploaded.")
        filename = list(uploaded.keys())[0]
        zip_path = os.path.join(UPLOAD_DIR, filename)
        with open(zip_path, "wb") as f:
            f.write(uploaded[filename])

    extract_dir = os.path.join(UPLOAD_DIR, "source_extracted")
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    all_chunks = []
    for root, _, files_list in os.walk(extract_dir):
        for fname in files_list:
            fpath = os.path.join(root, fname)
            if fname.endswith(".py"):
                all_chunks.extend(parse_python_file(fpath))
            elif fname.endswith((".js", ".ts", ".jsx", ".tsx", ".java")):
                all_chunks.extend(parse_generic_code_file(fpath))

    SESSION["chunks"].extend(all_chunks)
    print(f"Parsed {len(all_chunks)} code chunks from source archive.")
    for c in all_chunks[:5]:
        print(f"  {c.source_name}")
    return all_chunks

print("Source code ingestion utilities ready. Call upload_and_parse_source_code() to upload a .zip.")

Source code ingestion utilities ready. Call upload_and_parse_source_code() to upload a .zip.


In [6]:
# CELL 6: Upload and parse OpenAPI/Swagger documentation (JSON or YAML)
# Each endpoint (path + method) becomes its own ArtifactChunk so retrieval
# can surface precise API context (params, request/response schema) per requirement.

def parse_openapi_spec(spec: dict, source_name: str) -> List[ArtifactChunk]:
    chunks = []
    paths = spec.get("paths", {})
    for path, methods in paths.items():
        for method, details in methods.items():
            if method.lower() not in ("get", "post", "put", "delete", "patch"):
                continue
            summary = details.get("summary", "")
            description = details.get("description", "")
            parameters = details.get("parameters", [])
            request_body = details.get("requestBody", {})
            responses = details.get("responses", {})

            text_parts = [
                f"{method.upper()} {path}",
                f"Summary: {summary}",
                f"Description: {description}",
                f"Parameters: {json.dumps(parameters)[:400]}",
                f"RequestBody: {json.dumps(request_body)[:400]}",
                f"Responses: {json.dumps(responses)[:400]}",
            ]
            chunks.append(ArtifactChunk(
                chunk_id=str(uuid.uuid4()),
                source_type="api_doc",
                source_name=f"{method.upper()} {path}",
                text="\n".join(text_parts)[:CHUNK_SIZE_CHARS],
                metadata={"path": path, "method": method.upper(), "file": source_name}
            ))
    return chunks

def upload_and_parse_api_docs() -> List[ArtifactChunk]:
    """
    Uploads an OpenAPI/Swagger file (.json or .yaml/.yml), parses it into
    per-endpoint ArtifactChunks, and appends them to SESSION['chunks'].
    """
    print("Please select your OpenAPI/Swagger file (.json, .yaml, or .yml)...")
    uploaded = colab_files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")

    filename = list(uploaded.keys())[0]
    filepath = os.path.join(UPLOAD_DIR, filename)
    with open(filepath, "wb") as f:
        f.write(uploaded[filename])

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()

    if filename.lower().endswith(".json"):
        spec = json.loads(raw)
    elif filename.lower().endswith((".yaml", ".yml")):
        spec = yaml.safe_load(raw)
    else:
        raise ValueError("Unsupported format. Please upload .json, .yaml, or .yml.")

    api_chunks = parse_openapi_spec(spec, filename)
    SESSION["chunks"].extend(api_chunks)
    print(f"Parsed {len(api_chunks)} API endpoint chunks from '{filename}'.")
    for c in api_chunks[:5]:
        print(f"  {c.source_name}")
    return api_chunks

print("OpenAPI/Swagger ingestion utilities ready. Call upload_and_parse_api_docs() to upload a spec.")

OpenAPI/Swagger ingestion utilities ready. Call upload_and_parse_api_docs() to upload a spec.


In [ ]:
!pip install -q "gradio-client>=2.0.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 4.44.0 requires gradio-client==1.3.0, but you have gradio-client 2.6.0 which is incompatible.


In [7]:
# CELL 7: Generate embeddings for requirements and all artifact chunks
# Uses Sentence-Transformers (BAAI/bge-large-en-v1.5 by default).

from sentence_transformers import SentenceTransformer

_embedding_model_cache = {"model": None}

def get_embedding_model() -> SentenceTransformer:
    """Lazy-loads and caches the sentence-transformers model (avoids reloading on every call)."""
    if _embedding_model_cache["model"] is None:
        print(f"Loading embedding model: {EMBEDDING_MODEL_NAME} (first load may take a minute)...")
        _embedding_model_cache["model"] = SentenceTransformer(EMBEDGIN_MODEL_NAME if 'EMBEDGIN_MODEL_NAME' in globals() else EMBEDDING_MODEL_NAME)
        print("Embedding model loaded.")
    return _embedding_model_cache["model"]

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE_CHARS, overlap: int = CHUNK_OVERLAP_CHARS) -> List[str]:
    """Simple sliding-window character chunker with overlap, used for long free text."""
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

def build_requirement_chunks(requirements: List[Any]) -> List[Any]:
    """
    Converts Requirement objects into ArtifactChunks so they can be embedded
    alongside code and API doc chunks in the same FAISS index.
    """
    req_chunks = []
    for req in requirements:
        for i, piece in enumerate(chunk_text(req.text)):
            req_chunks.append(ArtifactChunk(
                chunk_id=str(uuid.uuid4()),
                source_type="requirement",
                source_name=f"{req.req_id}" + (f"-part{i+1}" if i > 0 else ""),
                text=piece,
                metadata={"req_id": req.req_id, "source_file": req.source_file}
            ))
    return req_chunks

def embed_texts(texts: List[str]) -> np.ndarray:
    """Encodes a list of strings into L2-normalized embedding vectors (float32) for cosine similarity via inner product."""
    model = get_embedding_model()
    embeddings = model.encode(
        texts,
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True,  # normalize so FAISS inner-product == cosine similarity
        convert_to_numpy=True,
    )
    return embeddings.astype("float32")

def prepare_all_chunks_for_indexing() -> List[Any]:
    """
    Combines requirement chunks with previously ingested code/API-doc chunks
    (SESSION['chunks']) into one master list ready for embedding + indexing.
    """
    req_chunks = build_requirement_chunks(SESSION["requirements"])
    # Avoid duplicate requirement chunks if this is called more than once
    SESSION["chunks"] = [c for c in SESSION["chunks"] if c.source_type != "requirement"]
    SESSION["chunks"] = req_chunks + SESSION["chunks"]
    print(f"Total chunks ready for indexing: {len(SESSION['chunks'])} "
          f"(requirements={len(req_chunks)}, "
          f"code={sum(1 for c in SESSION['chunks'] if c.source_type == 'code')}, "
          f"api_docs={sum(1 for c in SESSION['chunks'] if c.source_type == 'api_doc')})")
    return SESSION["chunks"]

print("Embedding utilities ready. Call prepare_all_chunks_for_indexing() then embed_texts() before building the FAISS index (Cell 8).")

Embedding utilities ready. Call prepare_all_chunks_for_indexing() then embed_texts() before building the FAISS index (Cell 8).


In [8]:
# CELL 8: Build, save, and load the FAISS vector index
# Uses IndexFlatIP (inner product) over normalized embeddings, which is
# mathematically equivalent to cosine similarity search — simple, exact,
# and fast enough for typical single-project artifact volumes in Colab.

def build_faiss_index() -> faiss.Index:
    """
    Embeds every chunk in SESSION['chunks'] and builds a FAISS IndexFlatIP.
    Stores the index and a parallel chunk_id map in SESSION for later retrieval.
    """
    chunks = prepare_all_chunks_for_indexing()
    if not chunks:
        raise ValueError("No chunks available to index. Ingest requirements/code/API docs first.")

    texts = [c.text for c in chunks]
    embeddings = embed_texts(texts)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    SESSION["faiss_index"] = index
    SESSION["chunk_id_map"] = [c.chunk_id for c in chunks]

    print(f"FAISS index built: {index.ntotal} vectors, dimension={dimension}.")
    return index

def save_faiss_index(name: str = "project_index") -> None:
    """Persists the FAISS index and accompanying chunk metadata to disk (INDEX_DIR)."""
    if SESSION["faiss_index"] is None:
        raise ValueError("No FAISS index in memory to save. Build it first with build_faiss_index().")

    index_path = os.path.join(INDEX_DIR, f"{name}.faiss")
    meta_path = os.path.join(INDEX_DIR, f"{name}_meta.json")

    faiss.write_index(SESSION["faiss_index"], index_path)

    chunk_lookup = {c.chunk_id: asdict(c) for c in SESSION["chunks"]}
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump({
            "chunk_id_map": SESSION["chunk_id_map"],
            "chunk_lookup": chunk_lookup
        }, f)

    print(f"Index saved to {index_path}")
    print(f"Metadata saved to {meta_path}")

def load_faiss_index(name: str = "project_index") -> faiss.Index:
    """Loads a previously saved FAISS index and metadata back into SESSION."""
    index_path = os.path.join(INDEX_DIR, f"{name}.faiss")
    meta_path = os.path.join(INDEX_DIR, f"{name}_meta.json")

    if not (os.path.exists(index_path) and os.path.exists(meta_path)):
        raise FileNotFoundError(f"No saved index found with name '{name}' in {INDEX_DIR}.")

    index = faiss.read_index(index_path)
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    SESSION["faiss_index"] = index
    SESSION["chunk_id_map"] = meta["chunk_id_map"]
    SESSION["chunks"] = [ArtifactChunk(**v) for v in meta["chunk_lookup"].values()]

    print(f"Loaded FAISS index '{name}' with {index.ntotal} vectors.")
    return index

print("FAISS index utilities ready. Call build_faiss_index() after ingesting artifacts, then optionally save_faiss_index().")

FAISS index utilities ready. Call build_faiss_index() after ingesting artifacts, then optionally save_faiss_index().


In [9]:
# CELL 9: Retrieve relevant context chunks for a given requirement
# Performs a similarity search over the FAISS index and returns the top-k
# ArtifactChunks (across requirement/code/api_doc types), which are later
# fed into the LLM prompt for test generation.

def _chunk_by_id(chunk_id: str) -> Optional[ArtifactChunk]:
    for c in SESSION["chunks"]:
        if c.chunk_id == chunk_id:
            return c
    return None

def retrieve_context_for_requirement(
    requirement_text: str,
    top_k: int = TOP_K_RETRIEVAL,
    source_type_filter: Optional[List[str]] = None
) -> List[ArtifactChunk]:
    """
    Embeds the requirement text as a query and retrieves the top_k most similar
    chunks from the FAISS index. Optionally restrict to specific source types
    (e.g., ["code", "api_doc"]) to bias retrieval toward implementation context.
    """
    if SESSION["faiss_index"] is None:
        raise ValueError("FAISS index not built. Call build_faiss_index() first.")

    query_vec = embed_texts([requirement_text])
    # Retrieve a larger pool if filtering, then trim down after filtering
    search_k = top_k * 4 if source_type_filter else top_k
    search_k = min(search_k, SESSION["faiss_index"].ntotal)

    scores, indices = SESSION["faiss_index"].search(query_vec, search_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        chunk_id = SESSION["chunk_id_map"][idx]
        chunk = _chunk_by_id(chunk_id)
        if chunk is None:
            continue
        if source_type_filter and chunk.source_type not in source_type_filter:
            continue
        results.append(chunk)
        if len(results) >= top_k:
            break

    return results

def retrieve_context_for_all_requirements(
    top_k: int = TOP_K_RETRIEVAL
) -> Dict[str, List[ArtifactChunk]]:
    """
    Convenience function: runs retrieval for every requirement in SESSION,
    returning a dict of {req_id: [ArtifactChunk, ...]}. Useful for a quick
    sanity check before running full test generation.
    """
    results = {}
    for req in SESSION["requirements"]:
        retrieved = retrieve_context_for_requirement(req.text, top_k=top_k)
        results[req.req_id] = retrieved
    return results

def preview_retrieval(req_id: str, top_k: int = TOP_K_RETRIEVAL) -> None:
    """Debug helper: prints retrieved context for a single requirement by its req_id."""
    req = next((r for r in SESSION["requirements"] if r.req_id == req_id), None)
    if req is None:
        print(f"Requirement '{req_id}' not found.")
        return
    print(f"Requirement: {req.text}\n")
    retrieved = retrieve_context_for_requirement(req.text, top_k=top_k)
    for c in retrieved:
        print(f"  [{c.source_type}] {c.source_name} -> {c.text[:100].strip()}...")

print("RAG retrieval utilities ready. Call retrieve_context_for_requirement(req.text) once the FAISS index is built.")

RAG retrieval utilities ready. Call retrieve_context_for_requirement(req.text) once the FAISS index is built.


In [10]:
# CELL 10: LLM client wrapper (OpenAI-compatible, swappable)
# Wraps chat completion calls behind a single function so the backend can be
# swapped for a local Llama/Mistral server (e.g., via an OpenAI-compatible
# endpoint like vLLM or Ollama) by only changing this cell.

from openai import OpenAI

_llm_client_cache = {"client": None}

def get_llm_client() -> OpenAI:
    """
    Lazily creates and caches an OpenAI client.
    To point at a local/self-hosted model instead, change base_url here, e.g.:
        OpenAI(api_key="not-needed", base_url="http://localhost:11434/v1")
    """
    if _llm_client_cache["client"] is None:
        if not OPENAI_API_KEY:
            print("WARNING: OPENAI_API_KEY is not set. Set it in Cell 2 or as an env var before generating tests.")
        _llm_client_cache["client"] = OpenAI(api_key=OPENAI_API_KEY)
    return _llm_client_cache["client"]

def call_llm(
    system_prompt: str,
    user_prompt: str,
    model: str = LLM_MODEL_NAME,
    temperature: float = 0.3,
    max_retries: int = 3,
    retry_delay_seconds: float = 2.0
) -> str:
    """
    Sends a chat completion request and returns the raw text response.
    Retries on transient failures (network errors, rate limits) with backoff.
    """
    client = get_llm_client()
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                temperature=temperature,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            return response.choices[0].message.content or ""
        except Exception as e:
            last_error = e
            print(f"LLM call failed (attempt {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                time.sleep(retry_delay_seconds * attempt)

    raise RuntimeError(f"LLM call failed after {max_retries} attempts: {last_error}")

def extract_json_from_llm_output(raw_text: str) -> Any:
    """
    Robustly extracts a JSON object/array from LLM output, stripping markdown
    code fences (```json ... ```) if the model wraps its response in them.
    """
    cleaned = raw_text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        # Fallback: try to locate the first '[' or '{' and last matching bracket
        start_candidates = [i for i, ch in enumerate(cleaned) if ch in "[{"]
        end_candidates = [i for i, ch in enumerate(cleaned) if ch in "]}"]
        if start_candidates and end_candidates:
            start = start_candidates[0]
            end = end_candidates[-1] + 1
            try:
                return json.loads(cleaned[start:end])
            except json.JSONDecodeError as e:
                raise ValueError(f"Could not parse LLM output as JSON: {e}\nRaw output:\n{raw_text[:500]}")
        raise ValueError(f"No JSON structure found in LLM output.\nRaw output:\n{raw_text[:500]}")

print("LLM client wrapper ready. Ensure OPENAI_API_KEY is set before calling call_llm().")

LLM client wrapper ready. Ensure OPENAI_API_KEY is set before calling call_llm().


In [11]:
# CELL 11: Structured prompt templates for each test category
# Each prompt instructs the LLM to return ONLY a JSON array of test case objects
# with a fixed schema, so output can be parsed reliably into GeneratedTest records.

TEST_CATEGORIES = ["unit", "integration", "edge", "boundary", "security"]

CATEGORY_GUIDANCE = {
    "unit": (
        "Focus on isolated logic: individual functions/methods, input/output correctness, "
        "return values, exceptions raised, and mocking of dependencies."
    ),
    "integration": (
        "Focus on interactions between components/modules, API endpoint request-response flows, "
        "database interactions, and data passed across layers (frontend/backend/DB)."
    ),
    "edge": (
        "Focus on unusual or extreme inputs: empty inputs, nulls, very large payloads, "
        "unexpected data types, concurrent access, and rare user workflows."
    ),
    "boundary": (
        "Focus on limit values: minimum/maximum allowed lengths, numeric limits, "
        "off-by-one conditions, timeouts (e.g., the 10-second response requirement), and threshold values."
    ),
    "security": (
        "Focus on security concerns: authentication/authorization bypass, injection attacks "
        "(SQL/XSS/command), insecure data storage, input validation, and access control violations."
    ),
}

SYSTEM_PROMPT_TEMPLATE = """You are an expert Senior QA Automation Engineer specializing in generating \
precise, high-quality software test cases from requirements and contextual project artifacts \
(source code, API documentation, and related requirements).

You MUST respond with ONLY a valid JSON array and nothing else — no markdown, no commentary, \
no explanations before or after the JSON.

Each element of the array must be a JSON object with exactly these fields:
- "title": short descriptive test title (string)
- "steps": numbered test steps as a single string, steps separated by newline characters (string)
- "expected_result": the expected outcome of the test (string)

Generate {num_tests} distinct, non-redundant "{category}" test cases.
Category focus: {category_guidance}
"""

USER_PROMPT_TEMPLATE = """REQUIREMENT:
{requirement_text}

RETRIEVED PROJECT CONTEXT (source code, API docs, related requirements):
{context_block}

Generate {num_tests} "{category}" test cases strictly based on the requirement and context above. \
If the context does not fully cover the requirement, use reasonable, industry-standard QA judgment \
to fill gaps, but keep tests grounded in the requirement's intent.

Respond with ONLY the JSON array as specified.
"""

def format_context_block(chunks: List[ArtifactChunk]) -> str:
    """Formats retrieved chunks into a readable block for the LLM prompt, labeled by source type."""
    if not chunks:
        return "(No additional context retrieved. Base tests on the requirement text alone.)"
    lines = []
    for c in chunks:
        label = c.source_type.upper()
        lines.append(f"[{label}: {c.source_name}]\n{c.text.strip()}")
    return "\n\n".join(lines)

def build_prompts_for_category(
    requirement: Requirement,
    context_chunks: List[ArtifactChunk],
    category: str,
    num_tests: int = 3
) -> Dict[str, str]:
    """Builds the system and user prompts for a single (requirement, category) pair."""
    if category not in TEST_CATEGORIES:
        raise ValueError(f"Unknown category '{category}'. Must be one of {TEST_CATEGORIES}.")

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        num_tests=num_tests,
        category=category,
        category_guidance=CATEGORY_GUIDANCE[category],
    )
    user_prompt = USER_PROMPT_TEMPLATE.format(
        requirement_text=requirement.text,
        context_block=format_context_block(context_chunks),
        num_tests=num_tests,
        category=category,
    )
    return {"system": system_prompt, "user": user_prompt}

print(f"Prompt templates ready for categories: {TEST_CATEGORIES}")

Prompt templates ready for categories: ['unit', 'integration', 'edge', 'boundary', 'security']


In [12]:
# CELL 12: Orchestrate end-to-end test generation
# For each requirement: retrieve context -> for each category, prompt the LLM ->
# parse JSON -> build GeneratedTest records with full traceability
# (requirement_id + retrieved_context_ids) stored in SESSION['generated_tests'].

def generate_tests_for_requirement(
    requirement: Requirement,
    categories: List[str] = TEST_CATEGORIES,
    num_tests_per_category: int = 3,
    top_k: int = TOP_K_RETRIEVAL
) -> List[GeneratedTest]:
    """Runs retrieval + LLM generation across all requested categories for one requirement."""
    context_chunks = retrieve_context_for_requirement(requirement.text, top_k=top_k)
    context_ids = [c.chunk_id for c in context_chunks]

    generated = []
    for category in categories:
        prompts = build_prompts_for_category(requirement, context_chunks, category, num_tests_per_category)
        try:
            raw_output = call_llm(prompts["system"], prompts["user"])
            parsed = extract_json_from_llm_output(raw_output)
        except Exception as e:
            print(f"  [WARN] Generation failed for {requirement.req_id} / {category}: {e}")
            continue

        if not isinstance(parsed, list):
            print(f"  [WARN] Unexpected LLM output shape for {requirement.req_id} / {category}: not a list. Skipping.")
            continue

        for item in parsed:
            if not all(k in item for k in ("title", "steps", "expected_result")):
                continue
            generated.append(GeneratedTest(
                test_id=str(uuid.uuid4())[:8],
                requirement_id=requirement.req_id,
                category=category,
                title=item["title"],
                steps=item["steps"],
                expected_result=item["expected_result"],
                retrieved_context_ids=context_ids,
            ))

    return generated

def generate_all_tests(
    categories: List[str] = TEST_CATEGORIES,
    num_tests_per_category: int = 3,
    top_k: int = TOP_K_RETRIEVAL,
    progress_callback=None
) -> List[GeneratedTest]:
    """
    Runs generate_tests_for_requirement() across every requirement in SESSION,
    accumulating results into SESSION['generated_tests']. Optionally reports
    progress via progress_callback(current_index, total, req_id) — used by the Gradio UI.
    """
    if not SESSION["requirements"]:
        raise ValueError("No requirements loaded. Upload an SRS first (Cell 4).")
    if SESSION["faiss_index"] is None:
        raise ValueError("FAISS index not built. Run build_faiss_index() first (Cell 8).")

    all_tests = []
    total = len(SESSION["requirements"])

    for i, req in enumerate(SESSION["requirements"], start=1):
        print(f"[{i}/{total}] Generating tests for {req.req_id}: {req.text[:70]}...")
        tests = generate_tests_for_requirement(req, categories, num_tests_per_category, top_k)
        all_tests.extend(tests)
        print(f"    -> {len(tests)} tests generated.")
        if progress_callback:
            progress_callback(i, total, req.req_id)

    SESSION["generated_tests"] = all_tests
    print(f"\nTotal tests generated: {len(all_tests)} across {total} requirements.")
    return all_tests

print("Test generation orchestrator ready. Call generate_all_tests() after building the FAISS index.")

Test generation orchestrator ready. Call generate_all_tests() after building the FAISS index.


In [13]:
# CELL 13: Build requirement -> retrieved artifact -> test traceability report
# Produces a flat table (pandas DataFrame) linking each requirement to the
# tests generated for it and the specific artifact chunks that were retrieved
# as context, satisfying FR6 (traceability display).

def build_traceability_table() -> pd.DataFrame:
    """
    Builds a DataFrame with one row per generated test, showing:
    requirement id/text, test id/category/title, and the source artifacts
    (by name) that were retrieved as context for that requirement.
    """
    if not SESSION["generated_tests"]:
        raise ValueError("No generated tests found. Run generate_all_tests() first.")

    req_lookup = {r.req_id: r for r in SESSION["requirements"]}
    rows = []

    for test in SESSION["generated_tests"]:
        req = req_lookup.get(test.requirement_id)
        req_text = req.text if req else "(unknown requirement)"

        context_names = []
        for cid in test.retrieved_context_ids:
            chunk = _chunk_by_id(cid)
            if chunk:
                context_names.append(f"[{chunk.source_type}] {chunk.source_name}")

        rows.append({
            "Requirement ID": test.requirement_id,
            "Requirement Text": req_text,
            "Test ID": test.test_id,
            "Category": test.category,
            "Test Title": test.title,
            "Test Steps": test.steps,
            "Expected Result": test.expected_result,
            "Retrieved Context": "; ".join(context_names) if context_names else "(none)",
        })

    df = pd.DataFrame(rows)
    return df

def traceability_summary() -> pd.DataFrame:
    """
    Aggregated view: one row per requirement, showing how many tests were
    generated per category. Useful as a coverage-at-a-glance dashboard.
    """
    if not SESSION["generated_tests"]:
        raise ValueError("No generated tests found. Run generate_all_tests() first.")

    summary_rows = []
    for req in SESSION["requirements"]:
        req_tests = [t for t in SESSION["generated_tests"] if t.requirement_id == req.req_id]
        row = {"Requirement ID": req.req_id, "Requirement Text": req.text[:80]}
        for cat in TEST_CATEGORIES:
            row[cat.capitalize()] = sum(1 for t in req_tests if t.category == cat)
        row["Total"] = len(req_tests)
        summary_rows.append(row)

    return pd.DataFrame(summary_rows)

print("Traceability utilities ready. Call build_traceability_table() or traceability_summary() after generation.")

Traceability utilities ready. Call build_traceability_table() or traceability_summary() after generation.


In [14]:
# CELL 14: Export generated tests to CSV, PDF, and JSON
# PDF export uses fpdf2 with basic text wrapping and pagination so long test
# suites render cleanly. All exports are written to EXPORT_DIR and timestamped.

def export_tests_to_csv(filename: Optional[str] = None) -> str:
    """Exports the traceability table (all generated tests) to a CSV file. Returns the file path."""
    df = build_traceability_table()
    if filename is None:
        filename = f"generated_tests_{int(time.time())}.csv"
    filepath = os.path.join(EXPORT_DIR, filename)
    df.to_csv(filepath, index=False)
    print(f"CSV exported to: {filepath}")
    return filepath

def export_tests_to_json(filename: Optional[str] = None) -> str:
    """Exports raw generated test records (with full traceability metadata) to a JSON file."""
    if not SESSION["generated_tests"]:
        raise ValueError("No generated tests found. Run generate_all_tests() first.")
    if filename is None:
        filename = f"generated_tests_{int(time.time())}.json"
    filepath = os.path.join(EXPORT_DIR, filename)

    payload = [asdict(t) for t in SESSION["generated_tests"]]
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"JSON exported to: {filepath}")
    return filepath

class TestReportPDF(FPDF):
    """Custom PDF layout with a running header/footer for the test report."""
    def header(self):
        self.set_font("Helvetica", "B", 14)
        self.cell(0, 10, "AI Requirement-to-Test Generator - Test Report", ln=True, align="C")
        self.ln(2)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Page {self.page_no()}", align="C")

def _pdf_safe_text(text: str) -> str:
    """Replaces characters unsupported by FPDF's default Latin-1 core fonts."""
    if text is None:
        return ""
    return (
        text.replace("\u2019", "'").replace("\u2018", "'")
            .replace("\u201c", '"').replace("\u201d", '"')
            .replace("\u2013", "-").replace("\u2014", "-")
            .encode("latin-1", "replace").decode("latin-1")
    )

def export_tests_to_pdf(filename: Optional[str] = None) -> str:
    """Exports all generated tests, grouped by requirement, into a formatted PDF report."""
    if not SESSION["generated_tests"]:
        raise ValueError("No generated tests found. Run generate_all_tests() first.")
    if filename is None:
        filename = f"generated_tests_{int(time.time())}.pdf"
    filepath = os.path.join(EXPORT_DIR, filename)

    pdf = TestReportPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    req_lookup = {r.req_id: r for r in SESSION["requirements"]}
    tests_by_req: Dict[str, List[GeneratedTest]] = {}
    for t in SESSION["generated_tests"]:
        tests_by_req.setdefault(t.requirement_id, []).append(t)

    for req_id, tests in tests_by_req.items():
        req = req_lookup.get(req_id)
        pdf.set_font("Helvetica", "B", 12)
        pdf.set_fill_color(230, 230, 250)
        pdf.multi_cell(0, 8, _pdf_safe_text(f"{req_id}: {req.text if req else ''}"), fill=True)
        pdf.ln(1)

        for t in tests:
            pdf.set_font("Helvetica", "B", 10)
            pdf.multi_cell(0, 6, _pdf_safe_text(f"[{t.category.upper()}] {t.title} (ID: {t.test_id})"))
            pdf.set_font("Helvetica", "", 9)
            pdf.multi_cell(0, 5, _pdf_safe_text(f"Steps:\n{t.steps}"))
            pdf.multi_cell(0, 5, _pdf_safe_text(f"Expected Result: {t.expected_result}"))
            pdf.ln(3)
        pdf.ln(4)

    pdf.output(filepath)
    print(f"PDF exported to: {filepath}")
    return filepath

print("Export utilities ready: export_tests_to_csv(), export_tests_to_json(), export_tests_to_pdf().")

Export utilities ready: export_tests_to_csv(), export_tests_to_json(), export_tests_to_pdf().


In [18]:
# CELL 15: Gradio interface tying the entire pipeline together
# Tabs: (1) Upload artifacts, (2) Build index & generate tests, (3) Results table,
# (4) Traceability summary, (5) Downloads. Wraps all prior cell functions —
# no new business logic here, only UI plumbing and Gradio-safe file handling.

import gradio as gr

def ui_upload_srs(file_obj):
    if file_obj is None:
        return "No file uploaded.", pd.DataFrame()
    filepath = file_obj.name
    fname = os.path.basename(filepath)
    dest = os.path.join(UPLOAD_DIR, fname)
    shutil.copy(filepath, dest)

    if fname.lower().endswith(".pdf"):
        raw_text = extract_text_from_pdf(dest)
    elif fname.lower().endswith(".txt"):
        raw_text = extract_text_from_txt(dest)
    else:
        return "Unsupported SRS format. Please upload .pdf or .txt.", pd.DataFrame()

    reqs = split_requirements(raw_text, fname)
    SESSION["requirements"] = reqs
    df = pd.DataFrame([{"Requirement ID": r.req_id, "Text": r.text} for r in reqs])
    return f"Parsed {len(reqs)} requirements from '{fname}'.", df

def ui_upload_code(file_obj):
    if file_obj is None:
        return "No file uploaded (optional)."
    dest = os.path.join(UPLOAD_DIR, os.path.basename(file_obj.name))
    shutil.copy(file_obj.name, dest)
    chunks = upload_and_parse_source_code(zip_path=dest)
    return f"Parsed {len(chunks)} code chunks from source archive."

def ui_upload_api(file_obj):
    if file_obj is None:
        return "No file uploaded (optional)."
    dest = os.path.join(UPLOAD_DIR, os.path.basename(file_obj.name))
    shutil.copy(file_obj.name, dest)
    with open(dest, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()
    fname = os.path.basename(dest)
    if fname.lower().endswith(".json"):
        spec = json.loads(raw)
    elif fname.lower().endswith((".yaml", ".yml")):
        spec = yaml.safe_load(raw)
    else:
        return "Unsupported format. Please upload .json, .yaml, or .yml."
    api_chunks = parse_openapi_spec(spec, fname)
    SESSION["chunks"].extend(api_chunks)
    return f"Parsed {len(api_chunks)} API endpoint chunks from '{fname}'."

def ui_build_index():
    try:
        build_faiss_index()
        return f"FAISS index built successfully with {SESSION['faiss_index'].ntotal} vectors."
    except Exception as e:
        return f"Error building index: {e}"

def ui_generate_tests(selected_categories, num_per_category, progress=gr.Progress()):
    if not selected_categories:
        return "Please select at least one test category.", pd.DataFrame(), pd.DataFrame()
    try:
        def cb(i, total, req_id):
            progress(i / total, desc=f"Generating tests for {req_id} ({i}/{total})")

        generate_all_tests(
            categories=selected_categories,
            num_tests_per_category=int(num_per_category),
            progress_callback=cb
        )
        trace_df = build_traceability_table()
        summary_df = traceability_summary()
        status = f"Generated {len(SESSION['generated_tests'])} tests across {len(SESSION['requirements'])} requirements."
        return status, trace_df, summary_df
    except Exception as e:
        return f"Error during generation: {e}", pd.DataFrame(), pd.DataFrame()

def ui_export_csv():
    try:
        path = export_tests_to_csv()
        return path
    except Exception as e:
        gr.Warning(f"CSV export failed: {e}")
        return None

def ui_export_pdf():
    try:
        path = export_tests_to_pdf()
        return path
    except Exception as e:
        gr.Warning(f"PDF export failed: {e}")
        return None

def ui_export_json():
    try:
        path = export_tests_to_json()
        return path
    except Exception as e:
        gr.Warning(f"JSON export failed: {e}")
        return None


with gr.Blocks(title="AI Requirement-to-Test Generator") as demo:
    gr.Markdown("#  AI Requirement-to-Test Generator\nRAG-powered test case generation from SRS, source code, and API docs.")

    with gr.Tab("1. Upload Artifacts"):
        gr.Markdown("### Upload SRS (required)")
        srs_file = gr.File(label="SRS file (.pdf or .txt)")
        srs_btn = gr.Button("Parse SRS", variant="primary")
        srs_status = gr.Textbox(label="Status", interactive=False)
        srs_table = gr.Dataframe(label="Parsed Requirements", interactive=False)
        srs_btn.click(ui_upload_srs, inputs=srs_file, outputs=[srs_status, srs_table])

        gr.Markdown("### Upload Source Code (optional, .zip)")
        code_file = gr.File(label="Source code archive (.zip)")
        code_btn = gr.Button("Parse Source Code")
        code_status = gr.Textbox(label="Status", interactive=False)
        code_btn.click(ui_upload_code, inputs=code_file, outputs=code_status)

        gr.Markdown("### Upload OpenAPI/Swagger Spec (optional)")
        api_file = gr.File(label="OpenAPI/Swagger file (.json/.yaml/.yml)")
        api_btn = gr.Button("Parse API Docs")
        api_status = gr.Textbox(label="Status", interactive=False)
        api_btn.click(ui_upload_api, inputs=api_file, outputs=api_status)

    with gr.Tab("2. Build Index & Generate"):
        gr.Markdown("### Step 1: Build the RAG index over all uploaded artifacts")
        build_btn = gr.Button("Build FAISS Index", variant="primary")
        build_status = gr.Textbox(label="Index Status", interactive=False)
        build_btn.click(ui_build_index, outputs=build_status)

        gr.Markdown("### Step 2: Generate tests")
        categories_cb = gr.CheckboxGroup(
            choices=TEST_CATEGORIES, value=TEST_CATEGORIES, label="Test categories to generate"
        )
        num_tests_slider = gr.Slider(1, 6, value=3, step=1, label="Tests per category per requirement")
        gen_btn = gr.Button("Generate Tests", variant="primary")
        gen_status = gr.Textbox(label="Generation Status", interactive=False)

    with gr.Tab("3. Results"):
        results_table = gr.Dataframe(label="Generated Tests (full detail)", interactive=False, wrap=True)

    with gr.Tab("4. Traceability Summary"):
        summary_table = gr.Dataframe(label="Requirement -> Test Coverage Summary", interactive=False)

    gen_btn.click(
        ui_generate_tests,
        inputs=[categories_cb, num_tests_slider],
        outputs=[gen_status, results_table, summary_table]
    )

    with gr.Tab("5. Export"):
        gr.Markdown("### Download generated tests")
        with gr.Row():
            csv_btn = gr.Button("Export CSV")
            pdf_btn = gr.Button("Export PDF")
            json_btn = gr.Button("Export JSON")
        csv_file_out = gr.File(label="CSV Download")
        pdf_file_out = gr.File(label="PDF Download")
        json_file_out = gr.File(label="JSON Download")

        csv_btn.click(ui_export_csv, outputs=csv_file_out)
        pdf_btn.click(ui_export_pdf, outputs=pdf_file_out)
        json_btn.click(ui_export_json, outputs=json_file_out)

print("Gradio UI defined. Run Cell 16 to launch it.")

Gradio UI defined. Run Cell 16 to launch it.


In [ ]:
# CELL 16: Launch the Gradio app
# share=True generates a public URL, useful inside Colab where localhost isn't
# directly reachable from your browser session.

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7a7f3d462b857ad38d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
